# 5 · Versions and the palette

- A node changes by getting a new version, a `@node` of the same id that names the one it replaces. A saved graph keeps the version it names until it is upgraded. `Upgrade` says how a saved graph moves to the new version, and `Deprecation` marks a version as going away.
- `registry.describe()` is the palette, the model an editor draws its list of nodes from. Each node in it is a `NodeDescription`, the same model for a function node and a subgraph.

In [ ]:
import json

from conductor import Deprecation, Graph, NodeRegistry, Upgrade, make_graph, node


@node("echo", title="Echo", description="Hands back the text it is given.", deprecation=Deprecation(header="Use version 2", migration="Version 2 adds a prefix; existing graphs keep working."))
def echo_v1(text: str) -> str:
    return text


@node("echo", title="Echo", description="Hands back the text, with a prefix in front if you give one.", category="text", replaces=echo_v1, upgrade=Upgrade())
def echo(text: str, prefix: str = "") -> str:
    return f"{prefix}{text}"


registry = NodeRegistry()
registry.add(echo)

## Versions

The decorated function is the newest version, and only it goes on the registry. It keeps the older ones. A deprecation is a notice for an editor, not an error.

In [ ]:
print("versions:      ", list(echo.versions))
print("current:       ", echo.version)
print("v1 deprecated? ", echo.versions[1].deprecation)
print("v2 deprecated? ", echo.deprecation)
print("v1 inputs:     ", list(echo_v1.inputs))
print("v2 inputs:     ", list(echo.inputs))

## A saved graph, and its upgrade

A graph saved with version 1 still runs version 1, since compiling never upgrades. `registry.upgrade` gives the record moved on to the newest version, for the host to show the graph's author.

In [ ]:
saved = make_graph(echo_v1, constants={"echo.text": "hi"})
print("runs:    ", Graph.compile(saved, registry).run().outputs)

moved = registry.upgrade(saved, "echo")
print("upgraded:", moved.model_dump(mode="json")["nodes"]["echo"])
print("runs:    ", Graph.compile(moved, registry).run().outputs)

## The palette

`describe()` is a pydantic model, so `model_dump(mode="json")` writes it as the JSON an editor reads: each port with its type, its JSON schema, whether it is required, and its default.

In [ ]:
palette = registry.describe().nodes
print(json.dumps(palette[0].model_dump(mode="json"), indent=2))